# POD-DeepONet

In [ ]:
import os
import json
import h5py
import sys, pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, str(pathlib.Path("..").resolve()))
from utils.datasets import load
from models.pod import PODTrainer, PODConfig
from models.pod_deeponet import BranchNet, PODDeepONet, PODDeepONetConfig, PODDeepONetTrainer

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

def rel_l2(true, pred):
    return np.linalg.norm(true - pred, axis=1) / np.linalg.norm(true, axis=1)

### Configuration

In [ ]:
# Run
RUN_NAME = "pod_deeponet_nu0.01_v1"
RUN_DIR = os.path.join("../../TEMPO_results", RUN_NAME)

# Data
TRAIN_NU = 0.01
ALL_NU = [0.001, 0.01, 0.1, 1.0]
N_TRAIN = 9000
N_TEST = 500

# POD
MAX_MODES = 32

# Branch network
HIDDEN_DIM = 256
N_LAYERS = 4
SENSOR_STRIDE = 1

# Training
N_EPOCHS = 5000
BATCH_SIZE = 1024

# Misc
SEED = 39
N_VIZ = 3

os.makedirs(RUN_DIR, exist_ok=True)
C0, C1, C2 = plt.cm.tab10(0), plt.cm.tab10(1), plt.cm.tab10(2)
print(f"Run dir: {os.path.abspath(RUN_DIR)}")

*(Lu et al., 2022)*

**Problem.** Given initial condition $u_0(x) = u(x, 0)$, approximate the
solution operator $\mathcal{G}: u_0 \mapsto u(\cdot, \cdot)$ for a
time-dependent PDE on domain $x \in \Omega$, $t \in [0, T]$.

#### Phase 1: Proper Orthogonal Decomposition

Let $\{u^n\}_{n=1}^N$ be training trajectories. Each trajectory is represented
as a vector $\mathbf{s}^n \in \mathbb{R}^{N_t N_x}$ by flattening the
spatiotemporal grid.


Compute the mean field:
$$\bar{u} = \frac{1}{N} \sum_{n=1}^N \mathbf{s}^n \in \mathbb{R}^{N_t N_x}$$

Apply SVD to the centered snapshot matrix $S \in \mathbb{R}^{N \times N_t N_x}$:
$$S - \bar{u} = U \Sigma V^\top$$

Retain the $P$ leading right singular vectors as POD modes:
$$\Phi = V_{:, 1:P} \in \mathbb{R}^{N_t N_x \times P}$$

The projection coefficients for each training trajectory are:
$$\boldsymbol{\beta}^n = (\mathbf{s}^n - \bar{u})\, \Phi \in \mathbb{R}^P$$

The modes $\Phi$ are fixed after this step. $P$ is chosen so that the retained
modes explain a prescribed fraction of total variance

#### Phase 2: Branch Network

A fully connected network $\mathcal{B}_\theta: \mathbb{R}^m \to \mathbb{R}^P$
is trained to predict the POD coefficients from the initial condition sampled
at $m$ sensor points $\{x_j\}_{j=1}^m$:
$$\hat{\boldsymbol{\beta}} = \mathcal{B}_\theta\bigl(u_0(x_1), \ldots, u_0(x_m)\bigr)$$

Training minimizes the mean squared error against Phase 1 coefficients:
$$\mathcal{L}(\theta) = \frac{1}{N} \sum_{n=1}^N
\bigl\| \mathcal{B}_\theta(u_0^n) - \boldsymbol{\beta}^n \bigr\|_2^2$$

The POD modes $\Phi$ are not updated during this phase.

---

#### Prediction

For a new initial condition $u_0^*$, the full solution is approximated as:
$$\hat{u}(x, t) = \bar{u}(x, t) + \mathcal{B}_\theta(u_0^*)\, \Phi^\top$$

The result is reshaped from $\mathbb{R}^{N_t N_x}$ to the spatiotemporal grid
$(N_t, N_x)$. The branch is evaluated once, and reconstruction is a single
matrix-vector product 


### Data loading

In [ ]:
_local  = pathlib.Path("..") / "data" / f"Burgers_Nu{TRAIN_NU}.hdf5"
_server = pathlib.Path(os.path.expanduser("~/data/1D/Burgers/Train")) / f"1D_Burgers_Sols_Nu{TRAIN_NU}.hdf5"
path = str(_local) if _local.exists() else str(_server)
print(f"data: {path}")

with h5py.File(path, "r") as f:
    raw = f["tensor"][:N_TRAIN + N_TEST]
    if raw.ndim == 4:
        raw = raw[..., 0]
    x_np = f["x-coordinate"][:]
    t_np = f["t-coordinate"][:]

N_total, Nt, Nx = raw.shape
print(f"loaded: N={N_total}, Nt={Nt}, Nx={Nx}")

Train/test split and reshape to flat trajectories `s[n] in R^(Nt*Nx)`:

In [ ]:
tensor_train = raw[:N_TRAIN]
tensor_test  = raw[N_TRAIN:]
del raw

In [ ]:
s_traj = torch.tensor(tensor_train.reshape(N_TRAIN, -1), dtype=torch.float32).to(DEVICE)  # (N_TRAIN, Nt*Nx)
u0_train = torch.tensor(tensor_train[:, 0, :], dtype=torch.float32).to(DEVICE)  # (N_TRAIN, Nx)

x = torch.tensor(x_np[:, None], dtype=torch.float32).to(DEVICE)

Initial conditions and test targets:

In [ ]:
u0_test = torch.tensor(tensor_test[:, 0, :], dtype=torch.float32).to(DEVICE)  # (N_TEST, Nx)

s_test = torch.tensor(tensor_test.reshape(-1, Nx), dtype=torch.float32).to(DEVICE)  # (N_TEST*Nt, Nx)

t_test = torch.tensor(np.tile(t_np, N_TEST), dtype=torch.float32).to(DEVICE)

### Phase 1: POD

In [ ]:
m = len(range(0, Nx, SENSOR_STRIDE))

In [ ]:
trainer_pod = PODTrainer(PODConfig(max_modes=MAX_MODES))
history_pod = trainer_pod.train(s_traj, x=None, t=None)

In [ ]:
coeffs_np = trainer_pod.basis.coeffs.cpu().numpy()  # (N_TRAIN, P)
sigmas = coeffs_np.std(axis=0) * np.sqrt(N_TRAIN - 1)
energy = sigmas ** 2
cumvar = np.cumsum(energy) / energy.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(range(1, len(sigmas) + 1), sigmas, 'o-', color=C0, markersize=4, lw=1.5)
ax.set_xlabel('Mode index')
ax.set_ylabel('Singular value')
ax.set_title('Phase 1: singular value decay', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.plot(range(1, len(cumvar) + 1), cumvar, 'o-', color=C0, markersize=4, lw=1.5)
ax.axhline(99.99, color=C1, ls='--', lw=1.2, label='99.99%')
ax.set_xlabel('Mode index')
ax.set_ylabel('Cumulative variance (%)')
ax.set_title('Phase 1: cumulative variance explained', fontweight='bold')
ax.legend(framealpha=0.7)
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "pod_phase1.png"), dpi=150, bbox_inches="tight")
plt.show()

### Phase 2: Branch network training

In [ ]:
P = trainer_pod.basis.num_modes
branch = BranchNet(m=m, P=P, hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS).to(DEVICE)
model = PODDeepONet(trainer_pod.basis, branch).to(DEVICE)

In [ ]:
cfg = PODDeepONetConfig(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, sensor_stride=SENSOR_STRIDE)
trainer = PODDeepONetTrainer(model, cfg)
history_branch = trainer.train(u0_train)

### Evaluation

In [ ]:
def predict_batched(trainer, u0, batch_size=256):
    parts = []
    for i in range(0, len(u0), batch_size):
        with torch.no_grad():
            parts.append(trainer.predict(u0[i:i+batch_size]).cpu())
    return torch.cat(parts, dim=0).numpy()

# Train error
idx = torch.randperm(len(s_traj))[:2000]
err_train = rel_l2(
    s_traj[idx].cpu().numpy(),
    predict_batched(trainer, u0_train[idx])
)

# Test error
err_test = rel_l2(
    tensor_test.reshape(N_TEST, -1),
    predict_batched(trainer, u0_test)
)

print(f'Train | mean={err_train.mean():.4f}  median={np.median(err_train):.4f}  std={err_train.std():.4f}')
print(f'Test  | mean={err_test.mean():.4f}  median={np.median(err_test):.4f}  std={err_test.std():.4f}  p95={np.percentile(err_test, 95):.4f}')

### Metrics

In [ ]:
metrics = {
    "run_name":     RUN_NAME,
    "n_modes":      int(P),
    "n_train":      N_TRAIN,
    "n_test":       N_TEST,
    "train_mean":   float(err_train.mean()),
    "train_median": float(np.median(err_train)),
    "train_std":    float(err_train.std()),
    "test_mean":    float(err_test.mean()),
    "test_median":  float(np.median(err_test)),
    "test_std":     float(err_test.std()),
    "test_p95":     float(np.percentile(err_test, 95)),
}
print(metrics)

### Cross-$\nu$ generalization

In [ ]:
DATA_DIR = pathlib.Path(os.path.expanduser("~/data/1D/Burgers/Train"))

cross_nu_metrics = {}
for nu in ALL_NU:
    fpath = DATA_DIR / f"1D_Burgers_Sols_Nu{nu}.hdf5"
    if not fpath.exists():
        print(f"  nu={nu:.3f}: file not found, skipping")
        continue

    with h5py.File(fpath, "r") as f:
        raw_nu = f["tensor"][-N_TEST:]
        if raw_nu.ndim == 4:
            raw_nu = raw_nu[..., 0]

    u0_nu = torch.tensor(raw_nu[:, 0, :], dtype=torch.float32).to(DEVICE)
    s_nu  = raw_nu.reshape(N_TEST, -1)

    pred_nu = predict_batched(trainer, u0_nu)
    err_nu  = rel_l2(s_nu, pred_nu)

    cross_nu_metrics[nu] = {
        "mean":   float(err_nu.mean()),
        "median": float(np.median(err_nu)),
        "std":    float(err_nu.std()),
        "p95":    float(np.percentile(err_nu, 95)),
    }
    tag = " (trained)" if nu == TRAIN_NU else ""
    print(f"  nu={nu:.3f}{tag}: mean={err_nu.mean():.4f}  median={np.median(err_nu):.4f}  std={err_nu.std():.4f}")

metrics["cross_nu"] = cross_nu_metrics

In [ ]:
nu_labels = [f"$\\nu={nu:.3f}$" for nu in cross_nu_metrics]
means   = [v["mean"]   for v in cross_nu_metrics.values()]
medians = [v["median"] for v in cross_nu_metrics.values()]
p95s    = [v["p95"]    for v in cross_nu_metrics.values()]

x_pos = np.arange(len(nu_labels))
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(x_pos - 0.2, means,   0.35, label='Mean',   color=C0, alpha=0.85, linewidth=0)
ax.bar(x_pos + 0.15, medians, 0.35, label='Median', color=C1, alpha=0.85, linewidth=0)

trained_idx = list(cross_nu_metrics.keys()).index(0.01)
ax.axvline(trained_idx, color='gray', ls='--', lw=1.2, alpha=0.7, label='Trained on')

ax.set_xticks(x_pos)
ax.set_xticklabels(nu_labels)
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('POD-DeepONet — cross-$\\nu$ generalization', fontweight='bold')
ax.legend(framealpha=0.7)
ax.grid(True, ls='--', alpha=0.25, axis='y')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "cross_nu.png"), dpi=150, bbox_inches="tight")
plt.show()

### Reconstruction examples

In [ ]:
rng = np.random.default_rng(SEED)
idxs = rng.choice(N_TEST, size=N_VIZ, replace=False)

fig, axes = plt.subplots(N_VIZ, 3, figsize=(14, 3 * N_VIZ))
for row, idx in enumerate(idxs):
    pred = trainer.predict(u0_test[idx:idx+1]).reshape(Nt, Nx).cpu().numpy()
    true = tensor_test[idx]
    err  = np.abs(true - pred)
    vmax = np.abs(true).max()
    rl2  = np.linalg.norm(true - pred) / np.linalg.norm(true)

    for col, (arr, title, cmap, vmin, vm) in enumerate([
        (true, 'Ground Truth',   'RdBu_r',  -vmax, vmax),
        (pred, 'POD-DeepONet',   'RdBu_r',  -vmax, vmax),
        (err,  'Absolute Error', 'Oranges',  0,     err.max()),
    ]):
        ax = axes[row, col]
        im = ax.imshow(arr, aspect='auto', origin='lower', cmap=cmap,
                       extent=[x_np.min(), x_np.max(), t_np.min(), t_np.max()],
                       vmin=vmin, vmax=vm)
        if row == 0:
            ax.set_title(title, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f't   (rel L2={rl2:.3f})')
        if row == N_VIZ - 1:
            ax.set_xlabel('x')
        ax.spines[['top', 'right']].set_visible(False)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('POD-DeepONet: reconstruction examples', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "pod_reconstruction.png"), dpi=150, bbox_inches="tight")
plt.show()

### Error distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(err_test, bins=30, color=C0, alpha=0.8, linewidth=0)
ax.axvline(err_test.mean(),     color=C1, ls='--', lw=1.5, label=f'Mean {err_test.mean():.4f}')
ax.axvline(np.median(err_test), color=C2, ls='--', lw=1.5, label=f'Median {np.median(err_test):.4f}')
ax.set_xlabel('Relative $L^2$ error')
ax.set_ylabel('Count')
ax.set_title('Test error distribution', fontweight='bold')
ax.legend(framealpha=0.7)
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.plot(np.sort(err_test), color=C0, lw=1.5)
ax.axhline(err_test.mean(), color=C1, ls='--', lw=1.2, alpha=0.8)
ax.set_xlabel('Trajectory rank')
ax.set_ylabel('Relative $L^2$ error')
ax.set_title('Sorted test errors', fontweight='bold')
ax.grid(True, ls='--', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('POD-DeepONet — test set errors', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "pod_err_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

### Checkpoint

In [ ]:
torch.save({
    "model": model.state_dict(),
    "cfg": cfg,
    "metrics": metrics,
    "run_name": RUN_NAME,
}, os.path.join(RUN_DIR, "model.pt"))

with open(os.path.join(RUN_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved to {os.path.abspath(RUN_DIR)}")